# Dipole moment as a counter-ion signal: three tests

**Question.** Instead of structural similarity, can a molecule's **dipole moment** be used to sort counter-ions,
either as the *first filter* or as the *ranking signal itself*?

**Short answer from the run below: no.** The true partner's dipole is no closer to the drug's than a random ion's,
a dipole filter removes the right answer about as often as random cutting, and ranking by dipole alone scores at
random level.

## What this notebook does
1. Computes a dipole for every drug and every ion (**MMFF partial charges, 5 conformers, measured about the
   centre of mass** - the origin matters for charged species).
2. **Test A.** Is the true partner's dipole closer to the drug's than other ions' are? (50% = no signal)
3. **Test B.** Use dipole as the **first filter**: keep only the ions nearest in dipole, then rank the rest by
   similarity. Does the true partner survive, and does accuracy hold?
4. **Test C.** Use dipole **instead of similarity**: rank ions by closeness of dipole.

Same hidden-answer protocol as the rest of the project: each drug's own known pairs are hidden before ranking.

**Needs:** `IL_420_final_pKa_v2.csv`, `API_pKa_table.csv`, `List_B_known_pairs.csv`, `master_IL_ions.csv`.
CPU only, about 3 minutes.

In [ ]:
!pip install -q rdkit
import os, time, numpy as np, pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, MACCSkeys
RDLogger.DisableLog('rdApp.*')

NEEDED = ['IL_420_final_pKa_v2.csv', 'API_pKa_table.csv', 'List_B_known_pairs.csv', 'master_IL_ions.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    from google.colab import files
    print('Upload:', missing)
    for name in files.upload():
        for want in missing:
            stem, ext = os.path.splitext(want)
            if name.startswith(stem) and name.endswith(ext): os.replace(name, want)
print('files ready')

## Step 1 - The dipole calculation

A dipole is how far positive and negative charge sit apart: mu = sum of (charge x position). Charges are the
force-field (MMFF) partial charges; positions come from 5 generated 3D shapes, averaged. For an ion, the number
depends on where you put the origin, so every dipole is measured about the molecule's **centre of mass**.

In [ ]:
DEBYE = 4.80320                      # e x Angstrom -> Debye

def dipole(smi, n_conf=5, seed=42):
    m = Chem.MolFromSmiles(smi); mh = Chem.AddHs(m)
    if mh.GetNumAtoms() == 1: return 0.0                       # a single atom has no dipole about its own centre
    p = AllChem.ETKDGv3(); p.randomSeed = seed
    cids = list(AllChem.EmbedMultipleConfs(mh, numConfs=n_conf, params=p))
    if not cids:
        p.useRandomCoords = True; cids = list(AllChem.EmbedMultipleConfs(mh, numConfs=n_conf, params=p))
    if not cids: return np.nan
    props = AllChem.MMFFGetMoleculeProperties(mh)
    if props is not None:
        AllChem.MMFFOptimizeMoleculeConfs(mh, maxIters=500)
        q = np.array([props.GetMMFFPartialCharge(i) for i in range(mh.GetNumAtoms())])
    else:                                                       # no MMFF parameters: Gasteiger charges instead
        AllChem.UFFOptimizeMoleculeConfs(mh, maxIters=500); AllChem.ComputeGasteigerCharges(mh)
        q = np.nan_to_num(np.array([float(a.GetProp('_GasteigerCharge')) for a in mh.GetAtoms()]))
    mass = np.array([a.GetMass() for a in mh.GetAtoms()]); mus = []
    for c in cids:
        X = mh.GetConformer(c).GetPositions(); com = (mass[:, None] * X).sum(0) / mass.sum()
        mus.append(np.linalg.norm((q[:, None] * (X - com)).sum(0)) * DEBYE)
    return float(np.mean(mus))

print('acetate:', round(dipole('CC(=O)[O-]'), 1), 'D | choline:', round(dipole('C[N+](C)(C)CCO'), 1), 'D')

## Step 2 - Data, and dipoles for every drug and ion

In [ ]:
T = pd.read_csv('IL_420_final_pKa_v2.csv'); A = pd.read_csv('API_pKa_table.csv').set_index('api_smiles')
B = pd.read_csv('List_B_known_pairs.csv'); M = pd.read_csv('master_IL_ions.csv')
to_master = {}
for mid, ids in zip(M.master_id, M.list_A_ion_ids):
    if pd.notna(ids):
        for i in str(ids).split(','): to_master[i.strip()] = mid
B['ion_id'] = B.counter_ion_id.map(to_master)
POOL = T[~T.source.str.startswith('excluded')]                      # 417 usable ions
cat = POOL[POOL.ion_type == 'cation'].master_id.values; ani = POOL[POOL.ion_type == 'anion'].master_id.values
chol = M[M.smiles == 'C[N+](C)(C)CCO'].master_id.iloc[0]

# drugs that can be tested: a pKa and a role (acid = needs a cation, base = needs an anion)
apis = [s for s in B.api_smiles.unique() if s in A.index and pd.notna(A.api_pKa[s]) and A.role[s] in ('acid', 'base')]

def cached(path, build):
    if os.path.exists(path): return pd.read_csv(path)
    t = time.time(); df = build(); df.to_csv(path, index=False); print(f'  {path}: {len(df)} molecules in {time.time()-t:.0f}s'); return df
ion_mu  = cached('ion_dipoles.csv',  lambda: pd.DataFrame([dict(master_id=r.master_id, dipole=dipole(r.smiles)) for r in POOL.itertuples()])).set_index('master_id').dipole
drug_mu = cached('drug_dipoles.csv', lambda: pd.DataFrame([dict(api_smiles=s, dipole=dipole(s)) for s in apis])).set_index('api_smiles').dipole
apis = [s for s in apis if pd.notna(drug_mu.get(s))]                # one drug gets no 3D shape and is skipped
print(len(apis), 'drugs |', len(ion_mu), 'ions')
print('median dipole: cations %.1f D, anions %.1f D, drugs %.1f D' % (
      ion_mu[POOL[POOL.ion_type=='cation'].master_id].median(), ion_mu[POOL[POOL.ion_type=='anion'].master_id].median(), drug_mu[apis].median()))

## Step 3 - The three tests

For each drug: candidates are all ions of the opposite charge (241 cations for an acid drug, 176 anions for a base
drug). The similarity ranking is the project's recommender: the 15 nearest known drug-ion pairs (MACCS keys) vote,
weighted by similarity, with usage count breaking ties; the drug's own pairs are hidden.

In [ ]:
fp = {s: MACCSkeys.GenMACCSKeys(Chem.MolFromSmiles(s)) for s in B.api_smiles.unique()}
allapis = list(B.api_smiles.unique()); pop_all = B.ion_id.value_counts()
res, pct = [], []
for s in apis:
    ids = cat if A.role[s] == 'acid' else ani
    own = B[B.api_smiles == s]; others = B[B.api_smiles != s].copy()
    sims = dict(zip(allapis, DataStructs.BulkTanimotoSimilarity(fp[s], [fp[a] for a in allapis])))
    others['sim'] = others.api_smiles.map(sims)
    votes = others.nlargest(15, 'sim').groupby('ion_id').sim.sum()
    an  = votes.reindex(ids).fillna(0).values
    pop = pop_all.reindex(ids).fillna(0).values - own.ion_id.value_counts().reindex(ids).fillna(0).values
    dist = np.abs(ion_mu.reindex(ids).values - drug_mu[s])           # how far each ion's dipole is from the drug's
    def ranks(keys, keep=None):
        order = [i for i in np.lexsort(keys) if keep is None or keep[i]]     # last key is the primary one
        return {ids[i]: k + 1 for k, i in enumerate(order)}
    base, dip_only = ranks((-pop, -an)), ranks((-pop, dist))
    near = np.argsort(dist); flt = {}
    for f in (0.7, 0.5, 0.3):
        keep = np.zeros(len(ids), bool); keep[near[:int(round(f * len(ids)))]] = True
        flt[f] = ranks((-pop, -an), keep)
    for r in own.itertuples():
        t = r.ion_id
        if t not in set(ids): continue
        i = list(ids).index(t)
        pct.append(dict(chol=(t == chol), pct=(dist < dist[i]).mean() * 100))
        res.append(dict(chol=(t == chol), base=base.get(t, np.nan), dip_only=dip_only.get(t, np.nan),
                        **{f'f{f}': flt[f].get(t, np.nan) for f in flt}))
R, P = pd.DataFrame(res), pd.DataFrame(pct); H = R[~R.chol]
print(f'pairs tested: {len(R)} | hard (answer is not choline): {len(H)}')

In [ ]:
print('TEST A - is the true partner\'s dipole closer to the drug\'s than other ions\'?  (50% = pure chance)')
print(f'   average percentile of the true partner: {P.pct.mean():.0f}%   (hard cases only: {P[~P.chol].pct.mean():.0f}%)\n')

print('TEST B - dipole as the FIRST filter, then similarity ranking inside what is left')
print(f"   {'strategy':42s}{'top-5 all':>10}{'top-5 hard':>12}{'partner kept':>14}   random cut keeps")
def line(name, col, random_keep=None):
    a, h = R[col], H[col]
    print(f"   {name:42s}{100*(a<=5).mean():>9.1f}%{100*(h<=5).mean():>11.1f}%{100*a.notna().mean():>13.0f}%" + (f"   {random_keep}%" if random_keep else ''))
line('no filter (the recommender)', 'base')
for f in (0.7, 0.5, 0.3): line(f'keep the {int(f*100)}% closest in dipole', f'f{f}', int(f*100))

print('\nTEST C - dipole INSTEAD of similarity (closest dipole first, usage count breaks ties)')
line('dipole only', 'dip_only')
print('\nreference: random guessing is about 2% top-5; popularity alone about 38% (all) and 24% (hard)')

## What happened, in plain words

| Test | Result | Reading |
|---|---|---|
| **A** closeness of dipole | True partner sits at about the **48th percentile** (50% = chance) | Real partners are not closer in dipole than random ions |
| **B** first filter, keep 70% / 50% / 30% | Keeps **76% / 49% / 33%** of true partners | Random cutting would keep 70 / 50 / 30, so the filter is no better than chance |
| **B** accuracy after filtering | Top-5 falls from **43.5%** to 38.3% / 23.5% / 17.9% | Every filter makes the recommender worse |
| **C** dipole instead of similarity | Top-5 about **3%** | Random-level; popularity alone gets about 38% |

**Why it fails (chemistry).** A salt forms through proton transfer and the strong attraction between full
charges. A lone ion's dipole says nothing about acidity or basicity, depends on the origin and the conformer, and
is screened by water. Partner choice in the literature also follows habit (choline is 16% of pairs).

**Other dipole trials in this project (separate notebooks)**
- `Dipole_HBond_Fallback_Test.ipynb`: similar-dipole drugs vote for ions, on the 154 pairs whose drug has no
  similar known drug. MMFF dipole 25.3% top-5 against popularity 22.1% - not significant (95% interval -4.1 to +11.5).
- `xTB_pKa_and_Dipoles.ipynb`: the same test with GFN2-xTB dipoles. 15.6% - **worse** than the crude version.
  Those xTB dipoles were not measured about the centre of mass, so recompute them before reuse.
- Earlier exploration: shape and polarity matching (Fabian-style) gave 2 of 25 hard cases and 0 of 55 choline cases.

## Limits
- One dipole per lone ion, from force-field charges. The dipole of the **ion pair** (DFT or xTB) is untested and
  is the only version that is physically meaningful for pairing.
- 446 of the 447 testable pairs (one drug gets no 3D shape).
- Filtering by dipole *distance* is one design; a rule that removes only extreme dipoles was not tried.

**Better home for dipole:** Phase 2 permeability, where polarity genuinely matters (the Abraham dipolarity term).